# Chapter 4: GPT model

Code from Sebastian Raschka's *Build a Large Language Model (From Scratch)*. Each cell has the exercise, the book code it needs, and my code.

In [ ]:
# Setup
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
BOOK = ROOT / "resources/books/llm"
MODELS_DIR = str(ROOT / "work/llm/gpt2")
BOOK_CODE = BOOK / "ch04/01_main-chapter-code"
sys.path.insert(0, str(BOOK_CODE))


In [ ]:
# Exercise 4.1: Parameters in the feed forward versus attention module
# Calculate and compare the number of parameters in the feed forward module and in the
# multi-head attention module of a transformer block.

# Book code: ch04 gpt.py
from gpt import TransformerBlock

GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of layers
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}

block = TransformerBlock(GPT_CONFIG_124M)

# Alex's code starts here


In [ ]:
# Exercise 4.2: Initialize larger GPT models
# Besides the 124M model, initialize GPT-2 medium (emb_dim 1024, 24 layers, 16 heads),
# large (emb_dim 1280, 36 layers, 20 heads), and XL (emb_dim 1600, 48 layers, 25 heads).
# As a bonus, calculate each model's total parameter count. Change only the
# configuration, not the GPTModel code.

# Book code: ch04 gpt.py
from gpt import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of layers
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}

# Alex's code starts here


In [ ]:
# Exercise 4.3: Using separate dropout parameters
# The model uses one drop_rate everywhere. Change the code to use separate dropout
# values for the embedding layer, the shortcut connections, and the multi-head attention
# module.

# Book code: ch04 gpt.py
import torch
import torch.nn as nn
from gpt import MultiHeadAttention, LayerNorm, FeedForward

GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of layers
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

# Edit the book code above or copy it below.
# Alex's code starts here
